# WBSNet — Option-A Paper Run on Colab

End-to-end runner for the **Option-A paper scope**:

- 7 ablation variants (A1-A7) × seeds on **Kvasir-SEG**
- Full WBSNet on **CVC-ClinicDB** and **ISIC2018**
- U-Net baselines on Kvasir, ClinicDB, ISIC2018
- Generalization eval: Kvasir checkpoint → CVC-ColonDB
- Aggregation, significance tests, complexity, qualitative figures

**Recommended runtime:** Colab Pro+ A100 with background execution.
**Backup:** Colab Pro L4 for smoke tests or smaller runs.

Before running:
1. `Runtime → Change runtime type → A100 GPU` (or L4 if on Pro).
2. `Runtime → Background execution` (Pro+ only) — lets you close the laptop.
3. Have the processed dataset on Google Drive at `MyDrive/WBSNet_Dataset/` with subdirs `kvasir/`, `cvc_clinicdb/`, `cvc_colondb/`, `isic2018/`.
4. If continuing from the Kaggle seed-3407 run, keep the exported `wbsnet_paper_runs/` folder at `MyDrive/wbsnet_paper_runs/`.
5. The expected processed layout is split-based: `kvasir/train/images`, `kvasir/val/masks`, `isic2018/test/images`, `cvc_colondb/test/masks`, etc.

Each section is independent — re-run only what you need.


## 1. Environment check

In [ ]:
!nvidia-smi

## 2. Clone repo

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/MrArrogant2002/WBSNet-research-paper.git"
REPO_DIR = Path("/content/WBSNet-research-paper")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already at {REPO_DIR} — pulling latest")
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

## 3. Install dependencies

Colab already ships with CUDA-enabled PyTorch. We install only the rest of the pip stack to avoid breaking the Colab CUDA build.

In [ ]:
!pip install --quiet \
    PyYAML==6.0.1 tqdm==4.66.4 wandb==0.17.0 pandas==2.2.2 tabulate==0.9.0 matplotlib==3.8.4 \
    pywavelets==1.6.0 albumentations==1.4.21 opencv-python==4.9.0.80 \
    scipy==1.12.0 Pillow==10.3.0 \
    segmentation-models-pytorch==0.3.4 pytorch-wavelets==1.3.0
!pip install --quiet -e .

import torch
print("torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4. Verify repo structure

In [ ]:
!python3 scripts/verify_repo.py

## 5. Mount Google Drive and link dataset

If your dataset is laid out differently, edit `DRIVE_DATASET` and `DATASET_MAP` below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DATASET = Path("/content/drive/MyDrive/WBSNet_Dataset")
assert DRIVE_DATASET.exists(), f"Dataset not found at {DRIVE_DATASET}. Edit the path above."

DATA_ROOT = REPO_DIR / "data"
DATA_ROOT.mkdir(exist_ok=True)

DATASET_MAP = {
    "kvasir": "Kvasir-SEG",
    "cvc_clinicdb": "CVC-ClinicDB",
    "cvc_colondb": "CVC-ColonDB",
    "isic2018": "ISIC2018",
}
for src_name, dst_name in DATASET_MAP.items():
    src = DRIVE_DATASET / src_name
    dst = DATA_ROOT / dst_name
    if not src.exists():
        print(f"  skip: {src} not present on Drive")
        continue
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        print(f"  warn: {dst} exists and is not a symlink — leaving as-is")
        continue
    os.symlink(src, dst)
    print(f"  linked: {src} -> {dst}")

!ls -la data/

## 6. Restore previous outputs and import Kaggle seed-3407 artifacts

Run this before the paper pipeline. It restores any previous Colab output backup and imports usable Kaggle seed-3407 checkpoints so completed Kvasir/CVC work is skipped instead of retrained.

In [ ]:
DRIVE_OUTPUTS = Path("/content/drive/MyDrive/WBSNet_outputs")
LEGACY_RUNS = Path("/content/drive/MyDrive/wbsnet_paper_runs")

if DRIVE_OUTPUTS.exists():
    print(f"Restoring previous outputs from {DRIVE_OUTPUTS}")
    !mkdir -p outputs
    !rsync -a --info=progress2 {DRIVE_OUTPUTS}/ outputs/
else:
    print(f"No previous Colab output backup found at {DRIVE_OUTPUTS}")

if LEGACY_RUNS.exists():
    print(f"Importing legacy Kaggle artifacts from {LEGACY_RUNS}")
    !python3 scripts/import_legacy_paper_runs.py \
        --legacy-root {LEGACY_RUNS} \
        --output-root outputs \
        --verify-forward
else:
    print(f"No legacy Kaggle folder found at {LEGACY_RUNS}; continuing without import")

!test -d outputs && find outputs -path '*/checkpoints/best.pt' | sort | sed -n '1,80p' || true


## 7. (Optional) W&B login

Skip this cell to run with W&B offline. Set `WANDB_API_KEY` to log online.

In [ ]:
# import os; os.environ["WANDB_API_KEY"] = "YOUR_WANDB_KEY_HERE"
# !wandb login --relogin

## 8. Smoke test (1 epoch, batch 2)

Confirms the pipeline runs end-to-end. ~2 min on A100, ~3 min on L4.

In [ ]:
!python3 train.py --config configs/kvasir_wbsnet.yaml \
    --override train.epochs=1 train.batch_size=2 \
                dataset.split_strategy=pre_split_dirs \
                dataset.num_workers=2 dataset.prefetch_factor=2 \
                evaluation.compute_hd95=false evaluation.max_visualizations=0 \
                runtime.wandb.mode=offline \
                experiment.name=smoke_test \
                experiment.run_name=smoke_test


## 9. Run the full Option-A paper pipeline

**This is the headline cell.** It runs 12 training configs across the seeds you specify, then the ColonDB generalization eval, then aggregation/significance/complexity. Already-completed runs (whose `best.pt` exists) are skipped — safe to re-run.

**Continuation plan for the current evidence:**
1. Import the Kaggle seed-3407 outputs in Section 6.
2. Run `--seeds 3407 3408 3409`. The runner skips completed seed-3407 Kvasir/CVC checkpoints and runs the missing ISIC2018 WBSNet plus seeds 3408/3409.

**Pro+ tip:** enable Background Execution so the cell keeps running with the laptop closed.

In [ ]:
# Continuation paper run. Existing best.pt checkpoints are skipped automatically.
!python3 scripts/run_paper_optionA.py \
    --seeds 3407 3408 3409 \
    --override train.epochs=150 train.batch_size=16 train.grad_accum_steps=1 \
               dataset.split_strategy=pre_split_dirs \
               dataset.num_workers=2 dataset.prefetch_factor=2 \
               train.save_every=5 runtime.device=cuda runtime.wandb.mode=offline \
               evaluation.max_visualizations=8


## 10. Generate qualitative paper figures

After training, build the qualitative comparison panels for the paper.

In [ ]:
from pathlib import Path

candidates = sorted(
    Path("outputs").rglob("checkpoints/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
kvasir_runs = [p for p in candidates if "kvasir_wbsnet" in str(p) and "baseline" not in str(p)]
if not kvasir_runs:
    raise RuntimeError("No Kvasir WBSNet checkpoint found.")

BEST_CKPT = kvasir_runs[0]
print("Using checkpoint:", BEST_CKPT)
PRED_DIR = BEST_CKPT.parent.parent / "predictions"

# The processed Kvasir card may not include a test split, so use val for qualitative panels.
if not PRED_DIR.exists() or not any(PRED_DIR.iterdir()):
    !python3 predict.py --config configs/kvasir_wbsnet.yaml \
        --checkpoint {BEST_CKPT} --split val \
        --override dataset.split_strategy=pre_split_dirs \
                   dataset.num_workers=2 runtime.wandb.mode=offline \
                   evaluation.max_visualizations=8

!python3 scripts/make_paper_figures.py \
    --input-dir {PRED_DIR} --limit 8 --columns 2


## 11. Save outputs back to Drive

Colab disks reset on disconnect. Always rsync after any major run.

In [ ]:
DRIVE_BACKUP = Path("/content/drive/MyDrive/WBSNet_outputs")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
!rsync -a --info=progress2 outputs/ {DRIVE_BACKUP}/

## 12. Build the paper PDF (optional, can also be done locally)

In [ ]:
# Uncomment to install LaTeX and build the PDF inside Colab.
# !apt-get install -y --quiet texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-publishers
# !cd paper && pdflatex paper && bibtex paper && pdflatex paper && pdflatex paper